# Evaluation
## Testing datasets

In [1]:
from wikifin_rag.ingest import load_wikifin_data

In [2]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [3]:
data_gen_instructions = """
You emulate a student or young professional who has questions about personal finance.
Formulate {} questions this person might ask based on a document passage.
The passage should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [4]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
openai_client = OpenAI()

In [5]:
from wikifin_rag.evaluation_utils import llm_structured_retry
import json

In [6]:
def generate_document_ground_truth(doc, n=5):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions.format(n),
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [7]:
from concurrent.futures import ThreadPoolExecutor
from wikifin_rag.evaluation_utils import map_progress, calculate_total_cost
import pandas as pd
from wikifin_rag.config import PROJECT_ROOT
from pathlib import Path
import numpy as np

In [8]:
def generate_corpus_ground_truth(documents, n=5, file_path=PROJECT_ROOT / "data" / "evals" / "ground_truth-new.csv"):
    with ThreadPoolExecutor(max_workers=6) as pool:
        results = map_progress(pool, documents, lambda doc: generate_document_ground_truth(doc, n=n))

    ground_truth = []
    usages = []

    for records, usage in results:
        ground_truth.extend(records)
        usages.append(usage)

    # total_cost = calculate_total_cost(usages)

    file_path = Path(file_path)
    file_path.parent.mkdir(parents=True, exist_ok=True)
    df_ground_truth = pd.DataFrame(ground_truth)
    df_ground_truth.to_csv(file_path, index=False)


In [9]:
# number of documents to use
n_documents = 100

# number of question to generate per document
n_questions = 5

# file path for the generated ground truth dataset
ground_truth_file_path = PROJECT_ROOT / "data" / "evals" / "ground_truth-new.csv"

In [10]:
# TODO: change to True to force a ground truth data refresh
refresh_ground_truth = False

In [11]:
documents = load_wikifin_data()
ground_truth_docs = np.random.choice(documents, size=n_documents, replace=False)

In [12]:
import os

In [13]:
if refresh_ground_truth or not os.path.exists(ground_truth_file_path):
    generate_corpus_ground_truth(documents=ground_truth_docs, n=n_questions, file_path=ground_truth_file_path)

In [14]:
df_ground_truth = pd.read_csv(ground_truth_file_path)
ground_truth = df_ground_truth.to_dict(orient="records")

In [15]:
openai_client.close()

## Retrieval function evaluation
### Text Search

In [16]:
from wikifin_rag.evaluation_utils import compute_relevance_total, evaluate
from wikifin_rag.ingest import build_text_index

In [17]:
ts_index = build_text_index(documents=documents)

In [18]:
def text_search(query):
    boost_dict = {"title": 2.0, "section": 3.0, "content": 1.5}

    return ts_index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [19]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/500 [00:00<?, ?it/s]

In [20]:
ts_metrics = evaluate(ground_truth=ground_truth, search_function=text_search)

  0%|          | 0/500 [00:00<?, ?it/s]

In [21]:
openai_client.close()

In [22]:
ts_metrics

{'hit_rate': 0.544, 'mrr': 0.39396666666666663}